In [ ]:
# ==========================================================
# 1. Installation des dépendances (si nécessaire)
# ==========================================================
# Décommentez et exécutez cette cellule si vous n'avez pas installé ces bibliothèques
# !pip install transformers peft torch accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# ==========================================================
# 2. Définir les chemins du modèle de base et de l'adaptateur
# ==========================================================
BASE_MODEL_ID = "facebook/nllb-200-distilled-600M"
ADAPTER_ID = "Farid59/nllb-darija-lora-model"  # Votre adaptateur sur le Hub

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")

# ==========================================================
# 3. CHARGEMENT CORRECT DU MODÈLE + ADAPTATEUR
# ==========================================================

# Étape A : Charger le tokenizer ET le modèle de BASE original.
# C'est le gros modèle de plus de 2 Go.
print(f"Chargement du modèle de base : {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID)
print("Modèle de base chargé.")

# Étape B : Appliquer votre adaptateur LoRA par-dessus le modèle de base.
# C'est ici que la magie opère. `PeftModel` sait comment combiner les deux.
print(f"Application de l'adaptateur : {ADAPTER_ID}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
print("Adaptateur appliqué avec succès.")


# ==========================================================
# 4. Préparer le modèle final pour l'inférence
# ==========================================================
model.to(device)
model.eval() # Passer en mode évaluation (important pour la performance)
print("Modèle final prêt pour l'inférence.")


# ==========================================================
# 5. Processus de traduction (votre code était déjà correct ici)
# ==========================================================
text_fr = "je veux manger"
src_lang = "fra_Latn"
tgt_lang = "ary_Arab"

print(f"\nTraduction de : '{text_fr}'")

tokenizer.src_lang = src_lang
inputs = tokenizer(text_fr, return_tensors="pt").to(device)
forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

# `torch.no_grad()` désactive le calcul de gradient, ce qui rend l'inférence plus rapide.
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=100
    )

result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("👉 Traduction :", result)

Utilisation du device : cuda
Chargement du modèle de base : facebook/nllb-200-distilled-600M...
Modèle de base chargé.
Application de l'adaptateur : Farid59/nllb-darija-lora-model...
Adaptateur appliqué avec succès.
Modèle final prêt pour l'inférence.

Traduction de : 'je veux manger'
👉 Traduction : amaamaane


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Remplacez par le nom de votre dépôt sur le Hub
MODEL_ID = "facebook/nllb-200-distilled-600M" 

# Déterminer le périphérique (GPU si disponible, sinon CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilisation du périphérique : {device}")

# Charger le tokenizer et le modèle
print("Chargement du tokenizer et du modèle...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device) # <-- Envoyer le modèle sur le bon périphérique
print("Modèle et tokenizer chargés.")


def translate(text_to_translate, source_lang_code, target_lang_code):
    """
    Fonction de traduction qui gère la tokenization, la génération et le décodage.
    """
    # 1. Ajouter le code de la langue source au texte
    text_with_lang_code = f">>{source_lang_code}<< {text_to_translate}"
    
    # 2. Tokenizer le texte d'entrée
    inputs = tokenizer(text_with_lang_code, return_tensors="pt").to(device)
    
    # 3. Générer la traduction en forçant le début avec le code de la langue cible
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang_code),
        max_new_tokens=256 # Définir une longueur maximale pour éviter les traductions trop longues
    )
    
    # 4. Décoder les tokens générés pour obtenir le texte
    translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    
    return translated_text


# --- Exemple 1: Traduction Français -> Darija ---
text_fr = "Bonjour tout le monde. J'espère que vous allez bien."
print(f"\nTexte source (FR) : {text_fr}")
translation_dr = translate(text_fr, "fra_Latn", "ary_Arab")
print(f"Traduction (DR) : {translation_dr}")

# --- Exemple 2: Traduction Anglais -> Darija ---
text_en = "This is a test to check if the model is working correctly."
print(f"\nTexte source (EN) : {text_en}")
translation_dr_from_en = translate(text_en, "eng_Latn", "ary_Arab")
print(f"Traduction (DR) : {translation_dr_from_en}")

# --- Exemple 3: Traduction Darija -> Français ---
text_dr = "السلام عليكم، كيدايرين؟" 
print(f"\nTexte source (DR) : {text_dr}")
translation_fr = translate(text_dr, "ary_Arab", "fra_Latn")
print(f"Traduction (FR) : {translation_fr}")

Utilisation du périphérique : cuda
Chargement du tokenizer et du modèle...
Modèle et tokenizer chargés.

Texte source (FR) : Bonjour tout le monde. J'espère que vous allez bien.
Traduction (DR) : "هلا الجميع. أتمنى أن تكونوا بخير.

Texte source (EN) : This is a test to check if the model is working correctly.
Traduction (DR) : >> << هادي اختبار لتحقق من إن كان النموذج يعمل بشكل صحيح.

Texte source (DR) : السلام عليكم، كيدايرين؟
Traduction (FR) : Bonjour, vous voulez bien ?


In [7]:
from azure.cognitiveservices.speech import SpeechConfig, SpeechSynthesizer

# Configuration avec votre clé Azure (créez-la sur portal.azure.com)
speech_config = SpeechConfig(
    subscription="votre-clé-api-ici",  # Ex: "a1b2c3d4e5f6g7h8i9j0"
    region="eastus"                   # "eastus" ou "westeurope"
)

# Voix marocaine native (choisissez l'une ou l'autre)
speech_config.speech_synthesis_voice_name = "ar-MA-JamalNeural"  # Masculin
# speech_config.speech_synthesis_voice_name = "ar-MA-MounaNeural"  # Féminin

# Synthèse du texte
synthesizer = SpeechSynthesizer(speech_config)
result = synthesizer.speak_text_async("بغيت ناكول كسكس ف مرّاكش").get()

# Sauvegarde du fichier audio
with open("darija.wav", "wb") as audio_file:
    audio_file.write(result.audio_data)

print("✅ Fichier audio généré : darija.wav")

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


✅ Fichier audio généré : darija.wav
